### pgvector CRUD 실습 예제

이 노트북에서는 `langchain_postgres`를 사용하여 간단한 벡터 데이터의 생성, 조회, 수정, 삭제 과정을 학습합니다.

In [2]:
from dotenv import load_dotenv
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_postgres.vectorstores import PGVector
from langchain_core.documents import Document

# 1. 환경 변수 로드
load_dotenv(override=True)

PGVECTOR_ID = os.getenv("PGVECTOR_ID")
PGVECTOR_PW = os.getenv("PGVECTOR_PW")
PGVECTOR_HOST = os.getenv("PGVECTOR_HOST", "localhost")
PGVECTOR_PORT = os.getenv("PGVECTOR_PORT", "5432")
PGVECTOR_DB = os.getenv("PGVECTOR_DB")

connection = f'postgresql+psycopg://{PGVECTOR_ID}:{PGVECTOR_PW}@{PGVECTOR_HOST}:{PGVECTOR_PORT}/{PGVECTOR_DB}'

# 2. 임베딩 모델 설정
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# 3. 벡터스토어 초기화
collection_name = "crud_example_collection"
vectorstore = PGVector(
    embeddings=embeddings,
    collection_name=collection_name,
    connection=connection,
    use_jsonb=True,
)

#### 1. Create (데이터 생성)
`add_texts` 또는 `add_documents`를 사용하여 데이터를 벡터 스토어에 저장합니다.

In [3]:
print("--- Step 1: Create ---")
texts = [
    "강아지는 충성심이 강한 동물입니다.",
    "고양이는 독립적인 성향을 가진 동물입니다.",
    "햄스터는 작고 귀여운 반려동물입니다."
]
metadatas = [{"category": "mammal"}, {"category": "mammal"}, {"category": "small_animal"}]
ids = ["dog-001", "cat-001", "ham-001"]

# 데이터 추가
vectorstore.add_texts(texts=texts, metadatas=metadatas, ids=ids)
print("데이터 저장 완료!")

--- Step 1: Create ---
데이터 저장 완료!


#### 2. Read (데이터 조회)
`similarity_search`를 사용하여 가장 유사한 데이터를 찾아봅니다.

In [4]:
print("--- Step 2: Read ---")
query = "집에서 키우는 작은 동물이 뭐야?"
results = vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(results):
    print(f"[{i+1}] {doc.page_content} (Metadata: {doc.metadata})")

--- Step 2: Read ---
[1] 햄스터는 작고 귀여운 반려동물입니다. (Metadata: {'category': 'small_animal'})
[2] 강아지는 충성심이 강한 동물입니다. (Metadata: {'category': 'mammal'})


#### 3. Update (데이터 수정)
`add_texts`나 `add_documents`를 호출할 때 **이미 존재하는 `ids`**를 전달하면 내용이 업데이트(Upsert)됩니다.

In [5]:
print("--- Step 3: Update ---")
# dog-001 (강아지) 정보를 수정
updated_text = ["강아지는 충성심이 아주 강력하며 사람을 잘 따릅니다."]
vectorstore.add_texts(texts=updated_text, ids=["dog-001"], metadatas=[{"category": "mammal", "updated": True}])

# 수정 결과 확인
doc_check = vectorstore.similarity_search("강아지에 대해 알려줘", k=1)
print(f"수정된 내용: {doc_check[0].page_content}")
print(f"수정된 메타데이터: {doc_check[0].metadata}")

--- Step 3: Update ---
수정된 내용: 강아지는 충성심이 아주 강력하며 사람을 잘 따릅니다.
수정된 메타데이터: {'updated': True, 'category': 'mammal'}


#### 4. Delete (데이터 삭제)
`delete(ids=[...])`를 사용하여 특정 ID의 데이터를 삭제하거나 필터를 사용할 수 있습니다.

In [6]:
print("--- Step 4: Delete ---")

# 1. 햄스터 데이터 삭제 (ID 기반)
vectorstore.delete(ids=["ham-001"])
print("햄스터 데이터 삭제 완료.")

# 삭제 확인
results_after_del = vectorstore.similarity_search("햄스터", k=3)
print(f"햄스터 검색 결과 개수: {len(results_after_del)}")
for doc in results_after_del:
    if "햄스터" not in doc.page_content:
        print(f"남아있는 데이터: {doc.page_content[:15]}...")

# 2. 컬렉션 전체 초기화 (필요시)
# vectorstore.drop_tables()

--- Step 4: Delete ---
햄스터 데이터 삭제 완료.
햄스터 검색 결과 개수: 2
남아있는 데이터: 강아지는 충성심이 아주 강력...
남아있는 데이터: 고양이는 독립적인 성향을 가...
